In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import  LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
import pandas as pd
import seaborn as sns
df = pd.read_csv('loan_approval_dataset.csv')
df.head()


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [3]:
df.drop(columns ="loan_id",inplace = True)
df

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected
...,...,...,...,...,...,...,...,...,...,...,...,...
4264,5,Graduate,Yes,1000000,2300000,12,317,2800000,500000,3300000,800000,Rejected
4265,0,Not Graduate,Yes,3300000,11300000,20,559,4200000,2900000,11000000,1900000,Approved
4266,2,Not Graduate,No,6500000,23900000,18,457,1200000,12400000,18100000,7300000,Rejected
4267,1,Not Graduate,No,4100000,12800000,8,780,8200000,700000,14100000,5800000,Approved


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4269 entries, 0 to 4268
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0    no_of_dependents          4269 non-null   int64 
 1    education                 4269 non-null   object
 2    self_employed             4269 non-null   object
 3    income_annum              4269 non-null   int64 
 4    loan_amount               4269 non-null   int64 
 5    loan_term                 4269 non-null   int64 
 6    cibil_score               4269 non-null   int64 
 7    residential_assets_value  4269 non-null   int64 
 8    commercial_assets_value   4269 non-null   int64 
 9    luxury_assets_value       4269 non-null   int64 
 10   bank_asset_value          4269 non-null   int64 
 11   loan_status               4269 non-null   object
dtypes: int64(9), object(3)
memory usage: 400.3+ KB


In [5]:
df.describe()

,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
count,4269.000000,4.269000e+03,4.269000e+03,4269.000000,4269.000000,4.269000e+03,4.269000e+03,4.269000e+03,4.269000e+03
mean,2.498712,5.059124e+06,1.513345e+07,10.900445,599.936051,7.472617e+06,4.973155e+06,1.512631e+07,4.976692e+06
std,1.695910,2.806840e+06,9.043363e+06,5.709187,172.430401,6.503637e+06,4.388966e+06,9.103754e+06,3.250185e+06
min,0.000000,2.000000e+05,3.000000e+05,2.000000,300.000000,-1.000000e+05,0.000000e+00,3.000000e+05,0.000000e+00
25%,1.000000,2.700000e+06,7.700000e+06,6.000000,453.000000,2.200000e+06,1.300000e+06,7.500000e+06,2.300000e+06
50%,3.000000,5.100000e+06,1.450000e+07,10.000000,600.000000,5.600000e+06,3.700000e+06,1.460000e+07,4.600000e+06
75%,4.000000,7.500000e+06,2.150000e+07,16.000000,748.000000,1.130000e+07,7.600000e+06,2.170000e+07,7.100000e+06
max,5.000000,9.900000e+06,3.950000e+07,20.000000,900.000000,2.910000e+07,1.940000e+07,3.920000e+07,1.470000e+07


In [6]:
#data cleaning

In [7]:
df.isnull().sum()

no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
import numpy as np
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

print(df.shape)

(4173, 12)


In [10]:
df.columns = df.columns.str.strip()

print(df.columns)

Index(['no_of_dependents', 'education', 'self_employed', 'income_annum',
       'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value',
       'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value',
       'loan_status'],
      dtype='object')


In [11]:
le_education = LabelEncoder()
le_self_employed = LabelEncoder()
le_loan_status = LabelEncoder()
df['self_employed'] = le_self_employed.fit_transform(df['self_employed'])
df['education'] = le_education.fit_transform(df['education'])
df['loan_status'] = le_loan_status.fit_transform(df['loan_status'])

df

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
1,0,1,1,4100000,12200000,8,417,2700000,2200000,8800000,3300000,1
2,3,0,0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,1
3,3,0,0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,1
4,5,1,1,9800000,24200000,20,382,12400000,8200000,29400000,5000000,1
5,0,0,1,4800000,13500000,10,319,6800000,8300000,13700000,5100000,1
...,...,...,...,...,...,...,...,...,...,...,...,...
4264,5,0,1,1000000,2300000,12,317,2800000,500000,3300000,800000,1
4265,0,1,1,3300000,11300000,20,559,4200000,2900000,11000000,1900000,0
4266,2,1,0,6500000,23900000,18,457,1200000,12400000,18100000,7300000,1
4267,1,1,0,4100000,12800000,8,780,8200000,700000,14100000,5800000,0


In [12]:
df.head()

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
1,0,1,1,4100000,12200000,8,417,2700000,2200000,8800000,3300000,1
2,3,0,0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,1
3,3,0,0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,1
4,5,1,1,9800000,24200000,20,382,12400000,8200000,29400000,5000000,1
5,0,0,1,4800000,13500000,10,319,6800000,8300000,13700000,5100000,1


In [13]:
x_loan_amount = df[['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_term', 'cibil_score', 'residential_assets_value',
       'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']]

y_loan_amount = df['loan_amount']


In [14]:
X_train_loan_amount , X_test_loan_amount , y_train_loan_amount, y_test_loan_amount = train_test_split(x_loan_amount , y_loan_amount , test_size = 0.2 , random_state = 42)
X_train_loan_amount

,no_of_dependents,education,self_employed,income_annum,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
567,5,0,1,3000000,12,441,8500000,2500000,7300000,2000000
3140,1,0,1,3800000,12,670,5200000,4700000,12200000,3800000
2458,2,0,0,2500000,20,547,800000,1200000,5200000,2300000
2002,3,0,0,5500000,14,853,7300000,3000000,15500000,3100000
1992,0,1,1,7300000,12,851,3300000,3400000,24800000,5300000
...,...,...,...,...,...,...,...,...,...,...
3520,3,1,0,6900000,4,650,900000,13400000,15500000,6400000
478,3,1,1,3600000,8,790,5400000,5900000,9800000,5200000
3162,1,0,1,9400000,12,443,14100000,2700000,23200000,5600000
3854,0,1,1,3100000,8,614,1700000,4600000,11000000,3300000


In [15]:
rfr=RandomForestRegressor(
    n_estimators = 1000,
    random_state= 42
)
rfr.fit(X_train_loan_amount,y_train_loan_amount)
y_pred =rfr.predict(X_test)
score = r2_score(y_test,y_pred)
print(score)

NameError: name 'X_test' is not defined

In [ ]:
rfr.score(X_train,y_train)

In [ ]:
rfr.score(X_test, y_test)

In [ ]:
#classification

In [ ]:
df.head()

In [19]:
x_loan_status = df.iloc[:,:-1]
print(x_loan_status)
y_loan_status = df.iloc[:,-1]
y_loan_status 

      no_of_dependents  education  self_employed  income_annum  loan_amount  \
1                    0          1              1       4100000     12200000   
2                    3          0              0       9100000     29700000   
3                    3          0              0       8200000     30700000   
4                    5          1              1       9800000     24200000   
5                    0          0              1       4800000     13500000   
...                ...        ...            ...           ...          ...   
4264                 5          0              1       1000000      2300000   
4265                 0          1              1       3300000     11300000   
4266                 2          1              0       6500000     23900000   
4267                 1          1              0       4100000     12800000   
4268                 1          0              0       9200000     29700000   

      loan_term  cibil_score  residential_assets_va

1       1
2       1
3       1
4       1
5       1
       ..
4264    1
4265    0
4266    1
4267    0
4268    0
Name: loan_status, Length: 4173, dtype: int64

In [20]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test=train_test_split(x_loan_status,y_loan_status,test_size=0.20,random_state=42)
X_train

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
567,5,0,1,3000000,11100000,12,441,8500000,2500000,7300000,2000000
3140,1,0,1,3800000,8800000,12,670,5200000,4700000,12200000,3800000
2458,2,0,0,2500000,7000000,20,547,800000,1200000,5200000,2300000
2002,3,0,0,5500000,19800000,14,853,7300000,3000000,15500000,3100000
1992,0,1,1,7300000,22600000,12,851,3300000,3400000,24800000,5300000
...,...,...,...,...,...,...,...,...,...,...,...
3520,3,1,0,6900000,22700000,4,650,900000,13400000,15500000,6400000
478,3,1,1,3600000,8600000,8,790,5400000,5900000,9800000,5200000
3162,1,0,1,9400000,21000000,12,443,14100000,2700000,23200000,5600000
3854,0,1,1,3100000,8500000,8,614,1700000,4600000,11000000,3300000


In [24]:
from sklearn.linear_model import  LogisticRegression
model=LogisticRegression(max_iter=500)
model.fit(X_train, Y_train)
y_pred=model.predict(X_test)
print(accuracy_score(Y_test,y_pred))

0.8191616766467066


In [25]:
print(classification_report(y_pred,Y_test))

              precision    recall  f1-score   support

           0       0.93      0.80      0.86       592
           1       0.64      0.86      0.73       243

    accuracy                           0.82       835
   macro avg       0.79      0.83      0.80       835
weighted avg       0.85      0.82      0.83       835

